# Entrenar LoRA A y B en Colab T4 (Persona B)

Ejecutar las celdas **de arriba a abajo**. Antes: *Entorno de ejecución → Cambiar tipo → GPU T4*.

- Todo lo que importa (adapters, curvas, logs) se guarda en **Google Drive** (`MyDrive/dp-pr3/`), no en el disco de Colab.
- El entrenamiento corre en **segundo plano**: la celda 4 lo lanza y termina. Con la celda 5 se mira el avance.
- Colab gratis desconecta si cierras la pestaña o el PC se duerme. Deja la pestaña abierta y el PC sin suspender.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
REPO = "https://github.com/jcdiegolopez/DP-PR3.git"
!rm -rf /content/DP-PR3 && git clone -q $REPO /content/DP-PR3
%cd /content/DP-PR3
!git log --oneline -3
!pip install -q -r requirements.txt trl==1.16.0 datasets==5.1.0 matplotlib
!python -c "import torch, transformers, peft, trl; print(torch.__version__, transformers.__version__, peft.__version__, trl.__version__, torch.cuda.get_device_name(0))"

## Prueba rápida (≈ 5 min)
40 pares, una época, sin tocar Drive. Si falla (memoria, versiones), falla aquí y no a la hora 3 del entrenamiento.

In [ ]:
!PYTHONIOENCODING=utf-8 python scripts/entrenar.py --config B --limite 40 --epochs 1 --max-length 2688 --batch 2 --accum 8 --fp16 --salida /content/prueba --resultados /content/res_prueba 2>&1 | tail -25

## 4. Lanzar A y luego B en segundo plano
Config A, y al terminar, config B. Cada una deja su adapter en Drive apenas termina.

In [ ]:
import os, subprocess
D = "/content/drive/MyDrive/dp-pr3"
os.makedirs(D, exist_ok=True)
cmd = (
    "cd /content/DP-PR3 && export PYTHONIOENCODING=utf-8 && "
    "for C in A B; do "
    f"python scripts/entrenar.py --config $C --max-length 2688 --batch 2 --accum 8 --fp16 "
    f"--salida {D}/config_$C --resultados {D}/resultados > {D}/log_$C.txt 2>&1; "
    "done"
)
subprocess.Popen(["nohup", "bash", "-c", cmd], start_new_session=True)
print("Entrenamiento lanzado. Logs en", D)

## 5. Ver el avance (ejecutar cuando quieras)

In [ ]:
import glob
for f in sorted(glob.glob(D + "/log_*.txt")):
    print("=====", f)
    texto = open(f, encoding="utf-8", errors="replace").read().replace("\r", "\n")
    print("\n".join(texto.strip().splitlines()[-6:]))
!nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv

## 6. Al terminar
Descargar de Drive `dp-pr3/resultados/` (JSON y PNG) y copiarlos a `resultados/` del repo. Los adapters quedan en
`dp-pr3/config_A` y `dp-pr3/config_B`; de cada uno solo hacen falta `adapter_config.json` y `adapter_model.safetensors`.